# Bayesian Optimization Impact — Before/After Comparison

Trains **two SNN models on identical BCPO-selected features and the identical real test set** --
one with fixed, arbitrary hyperparameters, one with the hyperparameters found by Optuna's Bayesian
search in Phase 4 -- so the only variable between them is hyperparameter tuning itself.

Requires `algae_dataset_train.csv` and `algae_dataset_test.csv` from Phase 2.


In [ ]:
from google.colab import files
import os

# Upload the training and test CSV files
uploaded = files.upload()

# Set paths
TRAIN_CSV = '/content/algae_dataset_train.csv'
TEST_CSV  = '/content/algae_dataset_test.csv'

# Verify
print("\nChecking files...")

print("TRAIN_CSV:", os.path.exists(TRAIN_CSV))
print("TEST_CSV :", os.path.exists(TEST_CSV))

## 1. Config

In [ ]:
TRAIN_CSV = '/content/algae_dataset_train.csv'
TEST_CSV  = '/content/algae_dataset_test.csv'
RANDOM_SEED = 42

# BCPO settings (identical to Phase 3/4)
BCPO_N_AGENTS = 20
BCPO_N_ITER   = 40
BCPO_ALPHA    = 0.9
BCPO_CV_FOLDS = 3
BCPO_FITNESS_SAMPLE_CAP = 3000

# BEFORE: fixed, un-tuned defaults
BEFORE_CONFIG = dict(hidden_units=32, beta=0.9, lr=1e-3, num_steps=25)

# AFTER: winning hyperparameters found by the Optuna Bayesian search in Phase 4
AFTER_CONFIG = dict(hidden_units=8, beta=0.506, lr=0.00907, num_steps=38)

TRAIN_EPOCHS = 150

CHART_OUT_PATH = '/content/bayesian_before_after.png'

## 2. Install & imports

In [ ]:
!pip install snntorch scikit-learn -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

import snntorch as snn
from snntorch import surrogate

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 3. Load train/test, scale

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

non_feature_cols = ['longitude', 'latitude', 'cyan_dn', 'label', 'source_date', 'is_synthetic']
feature_columns = [c for c in train_df.columns if c not in non_feature_cols]
assert len(feature_columns) == 31

X_train_raw = train_df[feature_columns].values.astype(np.float64)
y_train = train_df['label'].values.astype(np.int64)
X_test_raw = test_df[feature_columns].values.astype(np.float64)
y_test = test_df['label'].values.astype(np.int64)

scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 4. BCPO feature selection (identical mechanism to Phase 3/4)

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def binarize(v, rng):
    return (rng.random(len(v)) < sigmoid(v)).astype(int)

_rng_sub = np.random.default_rng(RANDOM_SEED)
if len(X_train) > BCPO_FITNESS_SAMPLE_CAP:
    sub0 = _rng_sub.choice(np.where(y_train == 0)[0],
                            size=min(BCPO_FITNESS_SAMPLE_CAP // 2, (y_train == 0).sum()), replace=False)
    sub1 = _rng_sub.choice(np.where(y_train == 1)[0],
                            size=min(BCPO_FITNESS_SAMPLE_CAP // 2, (y_train == 1).sum()), replace=False)
    fitness_idx = np.concatenate([sub0, sub1])
else:
    fitness_idx = np.arange(len(X_train))

X_fit, y_fit = X_train[fitness_idx], y_train[fitness_idx]

def fitness_function(mask, X=X_fit, y=y_fit, alpha=BCPO_ALPHA, cv=BCPO_CV_FOLDS):
    sel = np.where(mask == 1)[0]
    if len(sel) == 0:
        return 1.0
    clf = KNeighborsClassifier(n_neighbors=3)
    scores = cross_val_score(clf, X[:, sel], y, cv=cv)
    return alpha * (1 - scores.mean()) + (1 - alpha) * (len(sel) / X.shape[1])

class BCPO:
    def __init__(self, n_agents=BCPO_N_AGENTS, n_iter=BCPO_N_ITER, dim=31, seed=RANDOM_SEED):
        self.n_agents, self.n_iter, self.dim = n_agents, n_iter, dim
        self.rng = np.random.default_rng(seed)

    def optimize(self):
        pos = self.rng.uniform(-1, 1, size=(self.n_agents, self.dim))
        fitness = np.array([fitness_function(binarize(pos[i], self.rng)) for i in range(self.n_agents)])
        bi = int(np.argmin(fitness))
        best_pos, best_fit = pos[bi].copy(), fitness[bi]

        for t in range(self.n_iter):
            T = 2 * (1 - t / self.n_iter)
            for i in range(self.n_agents):
                r = self.rng.random()
                a, b = self.rng.integers(0, self.n_agents, size=2)
                if r < 0.25:
                    new_pos = pos[i] + self.rng.standard_normal(self.dim) * (pos[a] - pos[b])
                elif r < 0.5:
                    new_pos = pos[i] + self.rng.random(self.dim) * (pos.mean(axis=0) - pos[i])
                elif r < 0.75:
                    new_pos = best_pos + T * self.rng.standard_normal(self.dim) * (best_pos - pos[i])
                else:
                    new_pos = best_pos + self.rng.random(self.dim) * (best_pos - pos[i]) * 0.5
                new_pos = np.clip(new_pos, -3, 3)
                new_bin = binarize(new_pos, self.rng)
                new_fit = fitness_function(new_bin)
                if new_fit < fitness[i]:
                    pos[i], fitness[i] = new_pos, new_fit
                    if new_fit < best_fit:
                        best_fit, best_pos = new_fit, new_pos.copy()

        mask = binarize(best_pos, self.rng)
        if mask.sum() == 0:
            mask[np.argmax(best_pos)] = 1
        return mask

mask = BCPO().optimize()
sel_idx = np.where(mask == 1)[0]
selected_features = [feature_columns[i] for i in sel_idx]
print(f'Selected {len(selected_features)}/31 features:', selected_features)

X_train_sel = X_train[:, sel_idx]
X_test_sel  = X_test[:, sel_idx]

## 5. SNN architecture and a reusable train/evaluate function

In [ ]:
class SNNClassifier(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs, beta):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid()
        self.fc1  = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2  = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x, num_steps):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        spk2_rec = []
        for _ in range(num_steps):
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_rec.append(spk2)
        return torch.stack(spk2_rec, dim=0)


def train_and_evaluate(config, X_train, y_train, X_test, y_test, epochs=TRAIN_EPOCHS, label='', seed=RANDOM_SEED):
    # Re-seed immediately before model creation so BEFORE and AFTER both start from an identical,
    # controlled random weight initialization -- otherwise whichever model trains second inherits
    # a different RNG state than the first, making the comparison unfair and non-reproducible.
    torch.manual_seed(seed)
    model = SNNClassifier(X_train.shape[1], config['hidden_units'], 2, config['beta']).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=config['lr'])
    loss_fn = nn.CrossEntropyLoss()

    Xtr = torch.tensor(X_train, dtype=torch.float32, device=device)
    ytr = torch.tensor(y_train, dtype=torch.long, device=device)
    Xte = torch.tensor(X_test, dtype=torch.float32, device=device)

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        spk_rec = model(Xtr, config['num_steps'])
        loss = loss_fn(spk_rec.sum(dim=0), ytr)
        loss.backward()
        opt.step()
        if (epoch + 1) % 30 == 0:
            print(f'  [{label}] epoch {epoch+1}/{epochs}  loss={loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        spk_rec_te = model(Xte, config['num_steps'])
        preds = spk_rec_te.sum(dim=0).argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    return acc, f1, model

## 6. Train BEFORE (fixed defaults) and AFTER (Bayesian-tuned)

In [ ]:
print('Training BEFORE model (fixed hyperparameters)...')
print('  Config:', BEFORE_CONFIG)
before_acc, before_f1, _ = train_and_evaluate(BEFORE_CONFIG, X_train_sel, y_train, X_test_sel, y_test, label='BEFORE')

print('\nTraining AFTER model (Bayesian-tuned hyperparameters)...')
print('  Config:', AFTER_CONFIG)
after_acc, after_f1, _ = train_and_evaluate(AFTER_CONFIG, X_train_sel, y_train, X_test_sel, y_test, label='AFTER')

print(f'\n{"":20s} {"Accuracy":>10s} {"F1":>10s}')
print(f'{"Before (fixed)":20s} {before_acc*100:9.2f}% {before_f1:10.4f}')
print(f'{"After (Bayesian)":20s} {after_acc*100:9.2f}% {after_f1:10.4f}')
print(f'\nImprovement: {(after_acc-before_acc)*100:+.2f} pts accuracy, {(after_f1-before_f1):+.4f} F1')

## 7. Plot the comparison

In [ ]:
labels = ['Test Accuracy', 'Test F1 Score']
before_vals = [before_acc, before_f1]
after_vals = [after_acc, after_f1]

x = np.arange(len(labels))
width = 0.32

fig, ax = plt.subplots(figsize=(7.2, 5.4), dpi=150)
b1 = ax.bar(x - width/2, before_vals, width, label='Before Bayesian Optimization (fixed hyperparameters)', color='#8FA9BD')
b2 = ax.bar(x + width/2, after_vals, width, label='After Bayesian Optimization (Optuna-tuned)', color='#065A82')

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h*100:.2f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 6), textcoords='offset points', ha='center', fontsize=13, fontweight='bold')

ax.set_ylim(min(before_vals) - 0.05, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=13)
ax.set_ylabel('Score', fontsize=12)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), fontsize=10, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', color='#E3E9ED', linewidth=1, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(CHART_OUT_PATH, dpi=150, bbox_inches='tight')
plt.show()

print('Saved chart to', CHART_OUT_PATH)

## Notes
- Both models train on the exact same BCPO-selected feature subset and are evaluated on the exact
  same real, held-out test set -- the only difference is hyperparameter choice, so the gap between
  them isolates the effect of Bayesian optimization specifically.
- If you want to verify the AFTER hyperparameters yourself rather than trusting the hardcoded
  values, re-run the full Optuna search from the Phase 4 notebook and copy its `study.best_params`
  into `AFTER_CONFIG` here.
